In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install wandb
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yasith8314 (yasith8314-university-of-moratuwa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Baseline Model

In [ ]:
!cp /content/drive/MyDrive/Data_Science_Project/full_cache/cache.pt /content/

In [ ]:
import os
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- W&B integration ----
import wandb

# ------------------------------------------------------------------
# CONFIG — adjust paths as needed
# ------------------------------------------------------------------
class Config:
    CACHE_PATH = "/content/cache.pt"
    CHECKPOINT_DIR = "/content/checkpoints/"

    EMBEDDING_DIM = 192

    # Training
    BATCH_SIZE = 4096          # embeddings only — can be huge
    EPOCHS = 200
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    TRIPLET_MARGIN = 0.25
    K_NEGATIVES = 16            # candidates for semi-hard mining

    # Regularization
    DROPOUT = 0.15             # added to MLP hidden layers

    # Early stopping
    EARLY_STOPPING_PATIENCE = 35
    SCHEDULER_PATIENCE = 5
    SCHEDULER_FACTOR = 0.5
    GRAD_CLIP = 1.0

    # Eval
    EVAL_BATCH_SIZE = 8192

    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ------------------------------------------------------------------
# MODEL
# ------------------------------------------------------------------
class BaselineModel(nn.Module):
    """
    Static 3-layer MLP: [noisy, enhanced] -> fused embedding.
    Dropout added on hidden layers for regularization.
    """
    def __init__(self, embedding_dim=192, dropout=0.25):
        super().__init__()
        self.fc1 = nn.Linear(2 * embedding_dim, embedding_dim)
        self.fc2 = nn.Linear(embedding_dim, embedding_dim)
        self.fc3 = nn.Linear(embedding_dim, embedding_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, noisy_emb, enhanced_emb):
        x = torch.cat([noisy_emb, enhanced_emb], dim=-1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        fused = self.fc3(x)
        return F.normalize(fused, p=2, dim=-1)

In [ ]:
# ------------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------------
def calculate_eer(pos_scores, neg_scores):
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0
    try:
        from sklearn.metrics import roc_curve
    except ImportError:
        raise ImportError("pip install scikit-learn")
    y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    y_score = np.concatenate([pos_scores, neg_scores])
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    return ((fpr[eer_idx] + fnr[eer_idx]) / 2.0) * 100

In [ ]:
def evaluate_gate(gate, cache, split_meta, device, seed=123, batch_size=8192):
    gate.eval()
    rng = np.random.RandomState(seed)

    noisy_t = torch.from_numpy(cache["noisy_emb"]).float().to(device)
    enh_t = torch.from_numpy(cache["enhanced_emb"]).float().to(device)

    valid_spk_keys = cache["clean_anchor"]
    pos_indices = []
    anchors = []
    for row_i, row in split_meta.iterrows():
        key = (row["language"], row["speaker"])
        if key in valid_spk_keys:
            pos_indices.append(row_i)
            anchors.append(torch.as_tensor(valid_spk_keys[key], dtype=torch.float32))

    if not pos_indices:
        return {"overall": {"gate_eer": 0.0, "noisy_eer": 0.0, "enhanced_eer": 0.0}}

    pos_indices = np.array(pos_indices)
    anchor_tensor = torch.stack(anchors).to(device)

    # Sample negatives: same language & noise type, different speaker
    neg_indices = np.zeros_like(pos_indices)
    valid_meta = split_meta.loc[pos_indices]
    langs = valid_meta["language"].values
    spks = valid_meta["speaker"].values
    noises = valid_meta["noise_type"].values

    lang_noise_to_spk_idx = defaultdict(list)
    for l, s, n, row_idx in zip(langs, spks, noises, pos_indices):
        lang_noise_to_spk_idx[(l, n)].append((s, row_idx))

    for i, (l, s, n) in enumerate(zip(langs, spks, noises)):
        group = lang_noise_to_spk_idx[(l, n)]
        other_candidates = [idx for (spk, idx) in group if spk != s]
        if not other_candidates:
            fallback = valid_meta[(valid_meta["language"] == l) & (valid_meta["speaker"] != s)]
            other_candidates = fallback.index.tolist() if not fallback.empty else [pos_indices[i]]
        neg_indices[i] = rng.choice(other_candidates)

    all_gate_pos, all_gate_neg = [], []
    all_noisy_pos, all_noisy_neg = [], []
    all_enh_pos, all_enh_neg = [], []

    with torch.no_grad():
        for start_idx in range(0, len(pos_indices), batch_size):
            end_idx = start_idx + batch_size
            b_pos = pos_indices[start_idx:end_idx]
            b_neg = neg_indices[start_idx:end_idx]
            b_anchor = anchor_tensor[start_idx:end_idx]

            b_fused_pos = gate(noisy_t[b_pos], enh_t[b_pos])
            b_fused_neg = gate(noisy_t[b_neg], enh_t[b_neg])

            all_gate_pos.extend(F.cosine_similarity(b_anchor, b_fused_pos).cpu().numpy())
            all_gate_neg.extend(F.cosine_similarity(b_anchor, b_fused_neg).cpu().numpy())
            all_noisy_pos.extend(F.cosine_similarity(b_anchor, noisy_t[b_pos]).cpu().numpy())
            all_noisy_neg.extend(F.cosine_similarity(b_anchor, noisy_t[b_neg]).cpu().numpy())
            all_enh_pos.extend(F.cosine_similarity(b_anchor, enh_t[b_pos]).cpu().numpy())
            all_enh_neg.extend(F.cosine_similarity(b_anchor, enh_t[b_neg]).cpu().numpy())

    return {
        "overall": {
            "gate_eer": calculate_eer(all_gate_pos, all_gate_neg),
            "noisy_eer": calculate_eer(all_noisy_pos, all_noisy_neg),
            "enhanced_eer": calculate_eer(all_enh_pos, all_enh_neg),
        }
    }

In [ ]:
# ------------------------------------------------------------------
# TRAINING: Triplet loss with semi-hard mining
# ------------------------------------------------------------------
def train_baseline_model(cache, config, device, checkpoint_path=None):
    meta = cache["meta"].reset_index(drop=True)
    gate = BaselineModel(embedding_dim=config.EMBEDDING_DIM, dropout=config.DROPOUT).to(device)

    valid_spk_keys = list(cache["clean_anchor"].keys())
    spk_to_idx = {k: i for i, k in enumerate(valid_spk_keys)}

    anchor_stack = torch.stack([
        torch.as_tensor(cache["clean_anchor"][k], dtype=torch.float32)
        for k in valid_spk_keys
    ]).to(device)

    optimizer = torch.optim.AdamW(gate.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=config.SCHEDULER_FACTOR, patience=config.SCHEDULER_PATIENCE
    )

    # Pre-load all embeddings to GPU
    noisy_t = torch.from_numpy(cache["noisy_emb"]).float().to(device)
    enh_t = torch.from_numpy(cache["enhanced_emb"]).float().to(device)

    lang_arr = meta["language"].to_numpy()
    spk_arr = meta["speaker"].to_numpy()
    valid_spk_set = set(valid_spk_keys)
    has_anchor = np.array([(l, s) in valid_spk_set for l, s in zip(lang_arr, spk_arr)])
    row_anchor_idx = np.array([spk_to_idx.get((l, s), -1) for l, s in zip(lang_arr, spk_arr)], dtype=np.int64)
    row_anchor_idx_t = torch.from_numpy(row_anchor_idx).to(device)

    train_idx = meta.index[(meta["split"] == "train") & has_anchor].to_numpy()
    val_meta = meta[meta["split"] == "val"]

    # Pre-build speaker groups for fast negative sampling
    by_speaker = meta.groupby(["language", "speaker"]).groups
    spk_rows = {k: np.array(list(v)) for k, v in by_speaker.items()}
    lang_speakers = {}
    for (lang, spk) in spk_rows:
        lang_speakers.setdefault(lang, []).append(spk)
    other_speakers_map = {}
    for lang, spks in lang_speakers.items():
        spks_arr = np.array(spks, dtype=object)
        for spk in spks:
            other_speakers_map[(lang, spk)] = spks_arr[spks_arr != spk]

    start_epoch = 0
    best_val_eer = float('inf')
    best_state = None
    epochs_no_improve = 0

    if checkpoint_path and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        gate.load_state_dict(ckpt["gate_state"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_val_eer = ckpt["best_val_eer"]
        best_state = ckpt.get("best_state")
        epochs_no_improve = ckpt.get("epochs_no_improve", 0)
        print(f"Resumed at epoch {start_epoch}, best val EER: {best_val_eer:.2f}%")

    K = config.K_NEGATIVES
    margin = config.TRIPLET_MARGIN
    batch_size = config.BATCH_SIZE

    print(f"Training samples: {len(train_idx)}")
    print(f"Validation samples: {len(val_meta)}")
    print(f"Speakers: {len(valid_spk_keys)}")
    print(f"Device: {device}")

    for epoch in range(start_epoch, config.EPOCHS):
        gate.train()
        rng = np.random.RandomState(42 + epoch)
        perm = rng.permutation(train_idx)
        epoch_losses = []

        for start in range(0, len(perm), batch_size):
            batch_rows = perm[start:start + batch_size]
            if len(batch_rows) < 2:
                continue

            optimizer.zero_grad()

            b_noisy = noisy_t[batch_rows]
            b_enh = enh_t[batch_rows]
            b_anchor = anchor_stack[row_anchor_idx_t[batch_rows]]

            # Positive fused embedding
            pos_fused = gate(b_noisy, b_enh)

            # Sample K negative candidates
            neg_rows = np.empty((len(batch_rows), K), dtype=np.int64)
            for bi, r in enumerate(batch_rows):
                lang = lang_arr[r]
                spk = spk_arr[r]
                others = other_speakers_map[(lang, spk)]
                for k in range(K):
                    neg_spk = others[rng.randint(len(others))]
                    cand = spk_rows[(lang, neg_spk)]
                    neg_rows[bi, k] = cand[rng.randint(len(cand))]

            neg_rows_flat = neg_rows.reshape(-1)
            neg_noisy = noisy_t[neg_rows_flat]
            neg_enh = enh_t[neg_rows_flat]

            # Compute negatives WITHOUT grad (for mining selection only)
            with torch.no_grad():
                neg_fused_flat = gate(neg_noisy, neg_enh)
                neg_fused = neg_fused_flat.view(len(batch_rows), K, -1)

                d_pos = 1 - F.cosine_similarity(b_anchor, pos_fused.detach(), dim=-1)
                d_negs = 1 - F.cosine_similarity(
                    b_anchor.unsqueeze(1).expand(-1, K, -1),
                    neg_fused, dim=-1
                )

                # Semi-hard: d_pos < d_neg < d_pos + margin
                semi_hard_mask = (d_negs > d_pos.unsqueeze(1)) & (d_negs < (d_pos + margin).unsqueeze(1))
                masked_d = torch.where(semi_hard_mask, d_negs, torch.full_like(d_negs, float("inf")))
                has_semi_hard = semi_hard_mask.any(dim=1)
                fallback_idx = d_negs.argmin(dim=1)
                semi_idx = torch.where(has_semi_hard, masked_d.argmin(dim=1), fallback_idx)

            # Select the mined negatives and compute loss WITH grad
            semi_idx_np = semi_idx.detach().cpu().numpy()
            sel_neg_rows = neg_rows[np.arange(len(batch_rows)), semi_idx_np]
            sel_neg_fused = gate(noisy_t[sel_neg_rows], enh_t[sel_neg_rows])

            d_pos_active = 1 - F.cosine_similarity(b_anchor, pos_fused, dim=-1)
            d_neg_active = 1 - F.cosine_similarity(b_anchor, sel_neg_fused, dim=-1)
            loss = F.relu(d_pos_active - d_neg_active + margin).mean()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(gate.parameters(), max_norm=config.GRAD_CLIP)
            optimizer.step()
            epoch_losses.append(loss.item())

        # Validation
        val_result = evaluate_gate(gate, cache, val_meta, device, batch_size=config.EVAL_BATCH_SIZE)
        val_eer = val_result["overall"]["gate_eer"]
        val_noisy = val_result["overall"]["noisy_eer"]
        val_enh = val_result["overall"]["enhanced_eer"]
        scheduler.step(val_eer)

        # ---- W&B: log metrics per epoch ----
        wandb.log({
            "train_loss": np.mean(epoch_losses),
            "val_gate_eer": val_eer,
            "val_noisy_eer": val_noisy,
            "val_enhanced_eer": val_enh,
            "epoch": epoch + 1,
            "lr": optimizer.param_groups[0]['lr'],
        })

        print(f"Epoch {epoch+1:03d}/{config.EPOCHS} | "
              f"loss={np.mean(epoch_losses):.4f} | "
              f"val_gate={val_eer:.2f}% | val_noisy={val_noisy:.2f}% | val_enh={val_enh:.2f}% | "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")

        if val_eer < best_val_eer:
            best_val_eer = val_eer
            best_state = {k: v.clone() for k, v in gate.state_dict().items()}
            epochs_no_improve = 0
            # Log the new best metric
            wandb.log({"best_val_gate_eer": best_val_eer})
            print(f"  *** New best val EER: {best_val_eer:.2f}% ***")
        else:
            epochs_no_improve += 1

        if checkpoint_path:
            torch.save({
                "epoch": epoch,
                "gate_state": gate.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val_eer": best_val_eer,
                "best_state": best_state,
                "epochs_no_improve": epochs_no_improve,
            }, checkpoint_path)

        if epochs_no_improve >= config.EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        gate.load_state_dict(best_state)

    print(f"\n{'='*60}")
    print(f"Best validation EER: {best_val_eer:.2f}%")
    print(f"{'='*60}")
    return gate

In [ ]:
# ------------------------------------------------------------------
# PER-CONDITION TEST EVALUATION
# ------------------------------------------------------------------
def test_per_condition(gate, cache, device):
    test_meta = cache["meta"][cache["meta"]["split"] == "test"]
    noise_types = sorted(test_meta["noise_type"].unique())
    snrs = sorted(test_meta["snr"].unique())

    rows = []
    for nt in noise_types:
        for snr in snrs:
            cond = test_meta[(test_meta["noise_type"] == nt) & (test_meta["snr"] == snr)]
            if cond.empty:
                continue
            r = evaluate_gate(gate, cache, cond, device, batch_size=Config.EVAL_BATCH_SIZE)
            rows.append({
                "noise_type": nt,
                "snr_db": snr,
                "noisy_eer": r["overall"]["noisy_eer"],
                "enhanced_eer": r["overall"]["enhanced_eer"],
                "gate_eer": r["overall"]["gate_eer"],
                "improvement": r["overall"]["noisy_eer"] - r["overall"]["gate_eer"],
            })

    df = pd.DataFrame(rows)
    print("\n" + "="*90)
    print("PER-CONDITION TEST RESULTS — PaperFusionMLP")
    print("="*90)
    print(df.to_string(index=False))

    avg_noisy = df["noisy_eer"].mean()
    avg_enh = df["enhanced_eer"].mean()
    avg_gate = df["gate_eer"].mean()

    print(f"\n{'='*90}")
    print(f"Average across all test conditions:")
    print(f"  Noisy only:      {avg_noisy:.2f}%")
    print(f"  Enhanced only:   {avg_enh:.2f}%")
    print(f"  Fusion gate:     {avg_gate:.2f}%")
    print(f"  Improvement:     {avg_noisy - avg_gate:+.2f}% vs noisy, {avg_enh - avg_gate:+.2f}% vs enhanced")
    print(f"{'='*90}")

    # ---- W&B: log the table as a W&B Table ----
    table = wandb.Table(dataframe=df)
    wandb.log({"test_results_table": table, "avg_noisy_eer": avg_noisy, "avg_enhanced_eer": avg_enh, "avg_gate_eer": avg_gate})

    return df

In [ ]:
def load_cache_safe(cache_path):
    """
    Load a cache file saved in any format (dict with 'meta' or 'meta_records',
    list of dicts, or a dict with 'meta' as list) and return a normalized dict.
    """
    data = torch.load(cache_path, map_location="cpu", weights_only=False)

    # If it's a list of dicts (rows only), convert to full cache
    if isinstance(data, list):
        print("Cache is a list of rows — reconstructing full cache...")
        rows = data
        meta = pd.DataFrame(rows)
        # We don't have embeddings, so this is incomplete — warn
        print("WARNING: This cache contains only metadata, not embeddings.")
        return {"meta": meta, "noisy_emb": None, "enhanced_emb": None,
                "quality_vec": None, "cos_dist": None, "abs_diff": None,
                "clean_anchor": None, "feature_names": None}

    if not isinstance(data, dict):
        raise TypeError(f"Unexpected cache type: {type(data)}")

    # Normalize 'meta'
    if "meta" in data:
        meta_data = data["meta"]
        if isinstance(meta_data, pd.DataFrame):
            meta = meta_data
        elif isinstance(meta_data, list):
            meta = pd.DataFrame(meta_data)
        else:
            raise TypeError(f"Unexpected meta type: {type(meta_data)}")
    elif "meta_records" in data:
        meta = pd.DataFrame(data["meta_records"])
    else:
        # No meta key found — print available keys for debugging
        print("Available keys in cache:", list(data.keys()))
        raise KeyError("No 'meta' or 'meta_records' key found in cache.")

    # Build normalized cache
    cache = {
        "meta": meta,
        "noisy_emb": data.get("noisy_emb"),
        "enhanced_emb": data.get("enhanced_emb"),
        "quality_vec": data.get("quality_vec"),
        "cos_dist": data.get("cos_dist"),
        "abs_diff": data.get("abs_diff"),
        "clean_anchor": data.get("clean_anchor"),
        "feature_names": data.get("feature_names"),
    }

    # If clean_anchor is stored as a list of pairs, reconstruct dict
    if cache["clean_anchor"] is None and "clean_anchor_keys" in data:
        keys = data["clean_anchor_keys"]
        vals = data["clean_anchor_vals"]
        if isinstance(keys, np.ndarray) and isinstance(vals, np.ndarray):
            cache["clean_anchor"] = {
                tuple(k): torch.from_numpy(v) if isinstance(v, np.ndarray) else v
                for k, v in zip(keys, vals)
            }

    # If feature_names is a list of strings, keep as list
    if cache["feature_names"] is None and "feature_names" in data:
        cache["feature_names"] = list(data["feature_names"])

    return cache

In [ ]:
def main():
    # ---- Safely finish any previous active run ----
    try:
        if wandb.run is not None:
            wandb.finish()
    except:
        pass

    # ---- Initialize W&B ----
    wandb.init(
        project="speaker-verification-fusion",
        reinit=True,
        config={
            "batch_size": Config.BATCH_SIZE,
            "epochs": Config.EPOCHS,
            "lr": Config.LR,
            "weight_decay": Config.WEIGHT_DECAY,
            "triplet_margin": Config.TRIPLET_MARGIN,
            "k_negatives": Config.K_NEGATIVES,
            "dropout": Config.DROPOUT,
            "early_stopping_patience": Config.EARLY_STOPPING_PATIENCE,
            "scheduler_patience": Config.SCHEDULER_PATIENCE,
            "scheduler_factor": Config.SCHEDULER_FACTOR,
            "grad_clip": Config.GRAD_CLIP,
            "embedding_dim": Config.EMBEDDING_DIM,
            "device": Config.DEVICE,
            "cache_path": Config.CACHE_PATH,
        }
    )

    print("Loading cache...")
    cache = load_cache_safe(Config.CACHE_PATH)
    print(f"Loaded {len(cache['meta'])} rows.")
    print(f"Splits: {cache['meta']['split'].value_counts().to_dict()}")

    # If embeddings are missing, we can't train — abort
    if cache["noisy_emb"] is None:
        raise RuntimeError("Cache does not contain embeddings. Please rebuild with a full cache.")

    device = torch.device(Config.DEVICE)
    checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, "paper_fusion_mlp_2.pt")

    gate = train_baseline_model(cache, Config, device, checkpoint_path=checkpoint_path)
    test_per_condition(gate, cache, device)

    # ---- Save final model as a W&B artifact ----
    final_path = os.path.join(Config.CHECKPOINT_DIR, "paper_fusion_mlp_final_2.pt")
    torch.save({"gate_state": gate.state_dict()}, final_path)
    print(f"\nSaved final model to {final_path}")

    artifact = wandb.Artifact("paper-fusion-mlp", type="model")
    artifact.add_file(final_path)
    wandb.log_artifact(artifact)

    wandb.finish()

main()

Loading cache...
Loaded 524044 rows.
Splits: {'train': 403591, 'test': 73110, 'val': 47343}
Training samples: 403591
Validation samples: 47343
Speakers: 1251
Device: cuda
Epoch 001/200 | loss=0.1528 | val_gate=20.69% | val_noisy=22.28% | val_enh=23.77% | lr=1.00e-03
  *** New best val EER: 20.69% ***
Epoch 002/200 | loss=0.1203 | val_gate=19.88% | val_noisy=22.28% | val_enh=23.77% | lr=1.00e-03
  *** New best val EER: 19.88% ***
Epoch 003/200 | loss=0.1145 | val_gate=19.27% | val_noisy=22.28% | val_enh=23.77% | lr=1.00e-03
  *** New best val EER: 19.27% ***
Epoch 004/200 | loss=0.1111 | val_gate=19.25% | val_noisy=22.28% | val_enh=23.77% | lr=1.00e-03
  *** New best val EER: 19.25% ***
Epoch 005/200 | loss=0.1088 | val_gate=18.94% | val_noisy=22.28% | val_enh=23.77% | lr=1.00e-03
  *** New best val EER: 18.94% ***
Epoch 006/200 | loss=0.1069 | val_gate=19.00% | val_noisy=22.28% | val_enh=23.77% | lr=1.00e-03
Epoch 007/200 | loss=0.1053 | val_gate=18.76% | val_noisy=22.28% | val_enh=23.

avg_enhanced_eer,▁
avg_gate_eer,▁
avg_noisy_eer,▁
best_val_gate_eer,█▆▄▄▃▃▂▂▂▂▂▂▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
lr,████████████▄▄▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train_loss,█▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_enhanced_eer,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_gate_eer,█▄▄▃▃▂▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_noisy_eer,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
avg_enhanced_eer,23.41198


In [ ]:
!cp /content/checkpoints/baseline_model.pt /content/drive/MyDrive/Data_Science_Project/Models/baseline_model/

In [ ]:
from google.colab import runtime
runtime.unassign()